In [1]:
# %% [markdown]
# # Deep Learning Preprocessing for Sequence Models
# 
# This script prepares data for GRU/LSTM/Transformer models.
# Unlike classical models that use aggregated features, deep learning models
# use raw hourly sequences to capture temporal patterns.
#
# Output: 3D tensors (n_patients, n_timesteps, n_features)

# %% Imports
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings

warnings.filterwarnings('ignore')

# %% Configuration
DATA_DIR = Path("../data/temporal-respiratory-support")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Sequence parameters
SEQUENCE_LENGTH = 24  # First 24 hours
MIN_HOURS = 4         # Minimum hours of data required
LOS_THRESHOLD = 4     # Days for binary classification
RANDOM_STATE = 42

print(f"Configuration:")
print(f"  Sequence length: {SEQUENCE_LENGTH} hours")
print(f"  Minimum hours required: {MIN_HOURS}")
print(f"  LOS threshold: {LOS_THRESHOLD} days")

# %% Define feature groups for sequences
# These are features that vary over time (measured hourly)
TEMPORAL_FEATURES = [
    # Vitals
    'heart_rate', 'sbp', 'dbp', 'mbp', 'sbp_ni', 'dbp_ni', 'mbp_ni',
    'temperature', 'spo2', 'glucose',
    
    # Respiratory status (binary)
    'invasive', 'noninvasive', 'highflow',
    
    # Ventilator settings
    'set_peep', 'total_peep', 'rr', 'set_rr', 'total_rr',
    'set_tv', 'total_tv', 'set_fio2',
    
    # Blood gas
    'calculated_bicarbonate', 'pCO2', 'pH', 'pO2', 'so2',
    
    # GCS
    'gcs', 'gcs_motor', 'gcs_verbal', 'gcs_eyes',
    
    # Interventions
    'vasopressor', 'crrt',
    
    # Severity
    'sepsis3', 'sofa_24hours'
]

# Static features (don't change over time)
# MATCHING YOUR EXISTING FEATURE LIST
STATIC_FEATURES = [
    'gender', 'anchor_age', 'race', 'insurance', 'language',
    'marital_status', 'first_careunit', 'elixhauser_vanwalraven',
    'height_inch', 'pbw_kg'
]

print(f"\nTemporal features: {len(TEMPORAL_FEATURES)}")
print(f"Static features: {len(STATIC_FEATURES)}")

# %% Load all patient files
def load_patient_file(filepath: Path) -> pd.DataFrame:
    """Load a single patient CSV file."""
    return pd.read_csv(filepath)

def get_all_patient_files(data_dir: Path, sample_size: int = None) -> list:
    """Get list of all patient CSV files."""
    all_files = []
    
    for folder in sorted(data_dir.iterdir()):
        if folder.is_dir() and folder.name.isdigit():
            csv_files = list(folder.glob("*.csv"))
            all_files.extend(csv_files)
    
    print(f"Found {len(all_files)} patient files")
    
    if sample_size:
        all_files = all_files[:sample_size]
        print(f"Using sample of {sample_size} patients")
    
    return all_files

# %% Extract sequences from patient data
def extract_sequence(df: pd.DataFrame, 
                     temporal_features: list,
                     sequence_length: int = 24) -> tuple:
    """
    Extract fixed-length sequence from patient data.
    Memory optimized with float32.
    
    Returns:
        - sequence: numpy array of shape (sequence_length, n_features)
        - static: dict of static features
        - targets: dict of target variables
        - valid: bool indicating if patient has enough data
    """
    # Check if patient has enough hours
    actual_hours = min(len(df), int(df['los'].iloc[0] * 24))
    
    if actual_hours < MIN_HOURS:
        return None, None, None, False
    
    # Get first sequence_length hours
    df_seq = df[df['hr'] < sequence_length].copy()
    
    # Extract available temporal features
    available_features = [f for f in temporal_features if f in df_seq.columns]
    
    # Create sequence array with float32 for memory efficiency
    sequence = np.zeros((sequence_length, len(available_features)), dtype=np.float32)
    for i, feat in enumerate(available_features):
        values = df_seq[feat].values.astype(np.float32)  # Convert to float32
        # Pad if necessary
        if len(values) < sequence_length:
            padded = np.full(sequence_length, np.nan, dtype=np.float32)
            padded[:len(values)] = values
            sequence[:, i] = padded
        else:
            sequence[:, i] = values[:sequence_length]
    
    # Extract static features
    static = {}
    for feat in STATIC_FEATURES:
        if feat in df.columns:
            static[feat] = df[feat].iloc[0]
        else:
            # Handle missing columns
            static[feat] = None
    
    # Extract targets
    targets = {
        'los_days': np.float32(df['los'].iloc[0]),
        'los_binary': np.int8(df['los'].iloc[0] >= LOS_THRESHOLD),  # int8 for binary
        'mortality': np.int8(df['death_outcome'].iloc[0] if 'death_outcome' in df.columns else 0),
        'subject_id': np.int32(df['subject_id'].iloc[0])  # int32 for IDs
    }
    
    return sequence, static, targets, True

# %% Process all patients
def process_all_patients(file_list: list, 
                         temporal_features: list,
                         sequence_length: int = 24) -> dict:
    """
    Process all patient files and extract sequences.
    
    Returns dict with:
        - sequences: (n_patients, sequence_length, n_features)
        - static_features: (n_patients, n_static_features)
        - targets: DataFrame with target variables
        - feature_names: list of temporal feature names
    """
    sequences = []
    static_list = []
    targets_list = []
    
    available_features = None
    
    for filepath in tqdm(file_list, desc="Processing patients"):
        df = load_patient_file(filepath)
        
        # Check inclusion criteria
        if df['anchor_age'].iloc[0] < 18:
            continue
        if df['los'].iloc[0] > 90:
            continue
            
        seq, static, targets, valid = extract_sequence(
            df, temporal_features, sequence_length
        )
        
        if valid:
            sequences.append(seq)
            static_list.append(static)
            targets_list.append(targets)
            
            # Track available features from first valid patient
            if available_features is None:
                available_features = [f for f in temporal_features if f in df.columns]
        del df # Free memory
    
    # Convert to arrays
    sequences = np.array(sequences)
    targets_df = pd.DataFrame(targets_list)
    
    # Convert static to array
    static_df = pd.DataFrame(static_list)
    
    print(f"\nProcessed {len(sequences)} valid patients")
    print(f"Sequence shape: {sequences.shape}")
    print(f"Static features shape: {static_df.shape}")
    
    return {
        'sequences': sequences,
        'static_df': static_df,
        'targets_df': targets_df,
        'temporal_features': available_features
    }

# %% Load and process data
print("\n" + "=" * 60)
print("LOADING AND PROCESSING DATA")
print("=" * 60)

# Get patient files (use sample for development, None for full)
patient_files = get_all_patient_files(DATA_DIR, sample_size=30000)

# Process all patients
data = process_all_patients(patient_files, TEMPORAL_FEATURES, SEQUENCE_LENGTH)

sequences = data['sequences']
static_df = data['static_df']
targets_df = data['targets_df']
temporal_features = data['temporal_features']

print(f"\nFinal dataset:")
print(f"  Sequences: {sequences.shape} (patients, hours, features)")
print(f"  Static features: {static_df.shape}")
print(f"  Targets: {targets_df.shape}")

# Check for demographic data
print(f"\nDemographic data availability:")
print(f"  Gender: {static_df['gender'].notna().sum()} / {len(static_df)} patients")
print(f"  Race: {static_df['race'].notna().sum()} / {len(static_df)} patients")
print(f"  Insurance: {static_df['insurance'].notna().sum()} / {len(static_df)} patients")

# %% Target Balance Analysis
print("\n" + "=" * 60)
print("TARGET CLASS BALANCE")
print("=" * 60)

# Calculate counts and percentages
counts = targets_df['los_binary'].value_counts().sort_index()
percentages = targets_df['los_binary'].value_counts(normalize=True).sort_index() * 100

balance_summary = pd.DataFrame({
    'Count': counts,
    'Percentage (%)': percentages
})
balance_summary.index = ['Short Stay (<4 days)', 'Long Stay (>=4 days)']

print(balance_summary)

# %% Handle missing values in sequences
print("\n" + "=" * 60)
print("HANDLING MISSING VALUES")
print("=" * 60)

def impute_sequences(sequences: np.ndarray, strategy: str = 'forward_fill') -> np.ndarray:
    """
    Impute missing values in sequences.
    
    Strategies:
        - forward_fill: Fill with last valid value (then backward fill remaining)
        - mean: Fill with feature mean
        - zero: Fill with zero
    """
    sequences = sequences.copy()
    n_patients, n_timesteps, n_features = sequences.shape
    
    missing_before = np.isnan(sequences).sum()
    
    if strategy == 'forward_fill':
        for i in tqdm(range(n_patients), desc="Imputing sequences"):
            df_temp = pd.DataFrame(sequences[i])
            sequences[i] = df_temp.ffill().bfill().fillna(0).values
    
    elif strategy == 'mean':
        # Calculate mean per feature across all patients and timesteps
        for j in range(n_features):
            feature_mean = np.nanmean(sequences[:, :, j])
            if np.isnan(feature_mean):
                feature_mean = 0
            mask = np.isnan(sequences[:, :, j])
            sequences[:, :, j][mask] = feature_mean
    
    else:  # zero
        sequences = np.nan_to_num(sequences, nan=0)
    
    missing_after = np.isnan(sequences).sum()
    
    # Final cleanup - any remaining NaNs get zero
    sequences = np.nan_to_num(sequences, nan=0)
    
    print(f"Missing values: {missing_before:,} → {missing_after:,} → 0")
    
    return sequences

# Calculate missingness before imputation
missing_pct = np.isnan(sequences).mean() * 100
print(f"Overall missingness before imputation: {missing_pct:.1f}%")

# Impute sequences
sequences_imputed = impute_sequences(sequences, strategy='forward_fill')

# %% Encode static features
print("\n" + "=" * 60)
print("ENCODING STATIC FEATURES")
print("=" * 60)

def encode_static_features(df: pd.DataFrame) -> tuple:
    """Encode categorical static features with memory optimization."""
    df = df.copy()
    encoders = {}
    
    # Encode gender
    if 'gender' in df.columns:
        le = LabelEncoder()
        df['gender'] = df['gender'].fillna('Unknown')
        df['gender_encoded'] = le.fit_transform(df['gender'].astype(str)).astype(np.int8)  # int8 for small categories
        encoders['gender'] = le
        print(f"Gender encoded: {dict(zip(le.classes_, le.transform(le.classes_)))}")
        df = df.drop('gender', axis=1)
    
    # Encode race
    if 'race' in df.columns:
        le = LabelEncoder()
        df['race'] = df['race'].fillna('Unknown')
        encoded = le.fit_transform(df['race'].astype(str))
        # Use int8 if < 128 categories, else int16
        df['race_encoded'] = encoded.astype(np.int8 if len(le.classes_) < 128 else np.int16)
        encoders['race'] = le
        print(f"Race categories: {len(le.classes_)} unique values")
        print(f"  Sample: {list(le.classes_[:5])}")
        df = df.drop('race', axis=1)
    
    # Encode insurance
    if 'insurance' in df.columns:
        le = LabelEncoder()
        df['insurance'] = df['insurance'].fillna('Unknown')
        df['insurance_encoded'] = le.fit_transform(df['insurance'].astype(str)).astype(np.int8)
        encoders['insurance'] = le
        print(f"Insurance encoded: {dict(zip(le.classes_, le.transform(le.classes_)))}")
        df = df.drop('insurance', axis=1)
    
    # Encode language
    if 'language' in df.columns:
        le = LabelEncoder()
        df['language'] = df['language'].fillna('Unknown')
        encoded = le.fit_transform(df['language'].astype(str))
        df['language_encoded'] = encoded.astype(np.int8 if len(le.classes_) < 128 else np.int16)
        encoders['language'] = le
        print(f"Language categories: {len(le.classes_)} unique values")
        df = df.drop('language', axis=1)
    
    # Encode marital_status
    if 'marital_status' in df.columns:
        le = LabelEncoder()
        df['marital_status'] = df['marital_status'].fillna('Unknown')
        df['marital_status_encoded'] = le.fit_transform(df['marital_status'].astype(str)).astype(np.int8)
        encoders['marital_status'] = le
        print(f"Marital status categories: {len(le.classes_)} unique values")
        df = df.drop('marital_status', axis=1)
    
    # Encode first_careunit
    if 'first_careunit' in df.columns:
        le = LabelEncoder()
        df['first_careunit'] = df['first_careunit'].fillna('Unknown')
        df['first_careunit_encoded'] = le.fit_transform(df['first_careunit'].astype(str)).astype(np.int8)
        encoders['first_careunit'] = le
        print(f"First care unit categories: {len(le.classes_)} unique values")
        df = df.drop('first_careunit', axis=1)
    
    # Convert numerical features to float32 for memory efficiency
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].fillna(df[col].median()).astype(np.float32)
        elif df[col].dtype == 'int64':
            # Check range to determine appropriate int type
            col_max = df[col].max()
            if col_max < 128:
                df[col] = df[col].fillna(df[col].median()).astype(np.int8)
            elif col_max < 32768:
                df[col] = df[col].fillna(df[col].median()).astype(np.int16)
            else:
                df[col] = df[col].fillna(df[col].median()).astype(np.int32)
    
    return df, encoders

static_encoded, static_encoders = encode_static_features(static_df)

# Ensure all columns are memory-optimized
print(f"\nMemory optimization check:")
for col in static_encoded.columns:
    dtype = static_encoded[col].dtype
    print(f"  {col}: {dtype}")

print(f"\nStatic features after encoding: {static_encoded.columns.tolist()}")
print(f"Shape: {static_encoded.shape}")

# %% Normalize features
print("\n" + "=" * 60)
print("NORMALIZING FEATURES")
print("=" * 60)

def normalize_sequences(sequences: np.ndarray, 
                        fit_data: np.ndarray = None) -> tuple:
    """
    Normalize sequences using StandardScaler.
    
    Reshapes (n_patients, n_timesteps, n_features) → (n_samples, n_features)
    for fitting, then reshapes back.
    """
    n_patients, n_timesteps, n_features = sequences.shape
    
    # Reshape to 2D
    sequences_2d = sequences.reshape(-1, n_features)
    
    if fit_data is not None:
        fit_2d = fit_data.reshape(-1, n_features)
        scaler = StandardScaler()
        scaler.fit(fit_2d)
    else:
        scaler = StandardScaler()
        scaler.fit(sequences_2d)
    
    # Transform
    sequences_normalized = scaler.transform(sequences_2d)
    
    # Reshape back to 3D
    sequences_normalized = sequences_normalized.reshape(n_patients, n_timesteps, n_features)
    
    return sequences_normalized, scaler

# %% Create train/val/test splits
print("\n" + "=" * 60)
print("CREATING DATA SPLITS")
print("=" * 60)

# Get targets
y_binary = targets_df['los_binary'].values
y_continuous = targets_df['los_days'].values
subject_ids = targets_df['subject_id'].values

# First split: train+val vs test
idx = np.arange(len(sequences_imputed))
idx_trainval, idx_test = train_test_split(
    idx, test_size=0.15, stratify=y_binary, random_state=RANDOM_STATE
)

# Second split: train vs val
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=0.176,  # 0.176 of 0.85 ≈ 0.15 of total
    stratify=y_binary[idx_trainval], random_state=RANDOM_STATE
)

print(f"Train: {len(idx_train)} ({len(idx_train)/len(idx)*100:.1f}%)")
print(f"Val: {len(idx_val)} ({len(idx_val)/len(idx)*100:.1f}%)")
print(f"Test: {len(idx_test)} ({len(idx_test)/len(idx)*100:.1f}%)")

# %% Normalize using training data only
print("\nNormalizing sequences using training statistics...")

# Fit scaler on training data
train_sequences = sequences_imputed[idx_train]
seq_scaler = StandardScaler()
train_2d = train_sequences.reshape(-1, train_sequences.shape[2])
seq_scaler.fit(train_2d)

# Transform all splits
def normalize_split(sequences, scaler):
    n_p, n_t, n_f = sequences.shape
    seq_2d = sequences.reshape(-1, n_f)
    seq_norm = scaler.transform(seq_2d)
    return seq_norm.reshape(n_p, n_t, n_f)

X_train_seq = normalize_split(sequences_imputed[idx_train], seq_scaler)
X_val_seq = normalize_split(sequences_imputed[idx_val], seq_scaler)
X_test_seq = normalize_split(sequences_imputed[idx_test], seq_scaler)

# Static features - normalize on training data
static_train = static_encoded.iloc[idx_train]
static_scaler = StandardScaler()
static_scaler.fit(static_train)

X_train_static = static_scaler.transform(static_encoded.iloc[idx_train]).astype(np.float32)
X_val_static = static_scaler.transform(static_encoded.iloc[idx_val]).astype(np.float32)
X_test_static = static_scaler.transform(static_encoded.iloc[idx_test]).astype(np.float32)

# Targets (use smaller dtypes)
y_train_binary = y_binary[idx_train].astype(np.int8)
y_val_binary = y_binary[idx_val].astype(np.int8)
y_test_binary = y_binary[idx_test].astype(np.int8)

y_train_cont = y_continuous[idx_train].astype(np.float32)
y_val_cont = y_continuous[idx_val].astype(np.float32)
y_test_cont = y_continuous[idx_test].astype(np.float32)

print(f"\nFinal shapes:")
print(f"  X_train_seq: {X_train_seq.shape}")
print(f"  X_val_seq: {X_val_seq.shape}")
print(f"  X_test_seq: {X_test_seq.shape}")
print(f"  X_train_static: {X_train_static.shape}")

# %% Create attention mask (for transformers)
print("\n" + "=" * 60)
print("CREATING ATTENTION MASKS")
print("=" * 60)

# For now, all timesteps are considered valid
mask_train = np.ones((len(idx_train), SEQUENCE_LENGTH))
mask_val = np.ones((len(idx_val), SEQUENCE_LENGTH))
mask_test = np.ones((len(idx_test), SEQUENCE_LENGTH))

print(f"Attention masks created: {mask_train.shape}")

# %% Save processed data
print("\n" + "=" * 60)
print("SAVING PROCESSED DATA")
print("=" * 60)

# Create comprehensive data dictionary
dl_data = {
    # Sequences (3D tensors)
    'X_train_seq': X_train_seq,
    'X_val_seq': X_val_seq,
    'X_test_seq': X_test_seq,
    
    # Static features (2D arrays)
    'X_train_static': X_train_static,
    'X_val_static': X_val_static,
    'X_test_static': X_test_static,
    
    # Attention masks
    'mask_train': mask_train,
    'mask_val': mask_val,
    'mask_test': mask_test,
    
    # Targets - binary
    'y_train_binary': y_train_binary,
    'y_val_binary': y_val_binary,
    'y_test_binary': y_test_binary,
    
    # Targets - continuous
    'y_train_cont': y_train_cont,
    'y_val_cont': y_val_cont,
    'y_test_cont': y_test_cont,
    
    # Metadata
    'temporal_features': temporal_features,
    'static_features': list(static_encoded.columns),
    'sequence_length': SEQUENCE_LENGTH,
    'n_temporal_features': len(temporal_features),
    'n_static_features': len(static_encoded.columns),
    
    # Scalers for inference
    'seq_scaler': seq_scaler,
    'static_scaler': static_scaler,
    'static_encoders': static_encoders,
    
    # Subject IDs for analysis
    'subject_ids_train': subject_ids[idx_train],
    'subject_ids_val': subject_ids[idx_val],
    'subject_ids_test': subject_ids[idx_test],
}

# Save as pickle
with open(OUTPUT_DIR / 'dl_data_splits.pkl', 'wb') as f:
    pickle.dump(dl_data, f)
print(f"Saved: {OUTPUT_DIR / 'dl_data_splits.pkl'}")

# Also save as numpy arrays for easier loading in PyTorch
np.savez_compressed(
    OUTPUT_DIR / 'dl_sequences_all.npz',
    X_train_seq=X_train_seq,
    X_val_seq=X_val_seq,
    X_test_seq=X_test_seq,
    X_train_static=X_train_static,
    X_val_static=X_val_static,
    X_test_static=X_test_static,
    y_train_binary=y_train_binary,
    y_val_binary=y_val_binary,
    y_test_binary=y_test_binary,
    y_train_cont=y_train_cont,
    y_val_cont=y_val_cont,
    y_test_cont=y_test_cont,
    mask_train=mask_train,
    mask_val=mask_val,
    mask_test=mask_test
)
print(f"Saved: {OUTPUT_DIR / 'dl_sequences_all.npz'}")

# %% Summary
print("\n" + "=" * 60)
print("DEEP LEARNING DATA SUMMARY")
print("=" * 60)

print(f"""
## Data Shapes

| Split | Sequences | Static | Labels |
|-------|-----------|--------|--------|
| Train | {X_train_seq.shape} | {X_train_static.shape} | {y_train_binary.shape} |
| Val | {X_val_seq.shape} | {X_val_static.shape} | {y_val_binary.shape} |
| Test | {X_test_seq.shape} | {X_test_static.shape} | {y_test_binary.shape} |

## Feature Information

- Temporal features: {len(temporal_features)}
- Static features: {len(static_encoded.columns)} (including demographics)
- Sequence length: {SEQUENCE_LENGTH} hours

## Static Features (in order):
{list(static_encoded.columns)}

Index mapping for static features after encoding:
- gender_encoded: 0
- anchor_age: 1  
- race_encoded: 2
- insurance_encoded: 3
- language_encoded: 4
- marital_status_encoded: 5
- first_careunit_encoded: 6
- elixhauser_vanwalraven: 7
- height_inch: 8
- pbw_kg: 9

## Class Distribution

- Train: {y_train_binary.mean()*100:.1f}% long stay
- Val: {y_val_binary.mean()*100:.1f}% long stay  
- Test: {y_test_binary.mean()*100:.1f}% long stay
""")

print("\n✅ Deep learning preprocessing complete with demographics!")

Configuration:
  Sequence length: 24 hours
  Minimum hours required: 4
  LOS threshold: 4 days

Temporal features: 34
Static features: 10

LOADING AND PROCESSING DATA
Found 50920 patient files
Using sample of 30000 patients


Processing patients: 100%|██████████| 30000/30000 [02:16<00:00, 219.90it/s]



Processed 29867 valid patients
Sequence shape: (29867, 24, 34)
Static features shape: (29867, 10)

Final dataset:
  Sequences: (29867, 24, 34) (patients, hours, features)
  Static features: (29867, 10)
  Targets: (29867, 4)

Demographic data availability:
  Gender: 29867 / 29867 patients
  Race: 29867 / 29867 patients
  Insurance: 29867 / 29867 patients

TARGET CLASS BALANCE
                      Count  Percentage (%)
Short Stay (<4 days)  23491       78.652024
Long Stay (>=4 days)   6376       21.347976

HANDLING MISSING VALUES
Overall missingness before imputation: 57.1%


Imputing sequences: 100%|██████████| 29867/29867 [00:01<00:00, 21596.51it/s]


Missing values: 13,905,499 → 0 → 0

ENCODING STATIC FEATURES
Gender encoded: {'F': 0, 'M': 1}
Race categories: 33 unique values
  Sample: ['AMERICAN INDIAN/ALASKA NATIVE', 'ASIAN', 'ASIAN - ASIAN INDIAN', 'ASIAN - CHINESE', 'ASIAN - KOREAN']
Insurance encoded: {'Medicaid': 0, 'Medicare': 1, 'Other': 2}
Language categories: 2 unique values
Marital status categories: 5 unique values
First care unit categories: 9 unique values

Memory optimization check:
  anchor_age: int8
  elixhauser_vanwalraven: float32
  height_inch: float32
  pbw_kg: float32
  gender_encoded: int8
  race_encoded: int8
  insurance_encoded: int8
  language_encoded: int8
  marital_status_encoded: int8
  first_careunit_encoded: int8

Static features after encoding: ['anchor_age', 'elixhauser_vanwalraven', 'height_inch', 'pbw_kg', 'gender_encoded', 'race_encoded', 'insurance_encoded', 'language_encoded', 'marital_status_encoded', 'first_careunit_encoded']
Shape: (29867, 10)

NORMALIZING FEATURES

CREATING DATA SPLITS
Trai